# * EJERCICIO 1 ❗: ANÁLISIS DE DATOS DE LA FÁBRICA [1p]
A partir de la función cargarCSV(), debes realizar un análisis de los datos de la fábrica utilizando el dataframe que esta función obtiene.

En concreto tienes que obtener los siguientes resultados:
1.   Guarda y muestra el valor de las coordenadas donde se encuentra la batería.
2.   Calcula la distancia entre cada casilla de la fábrica y la posición de la batería y guarda los valores en una nueva columna del dataframe llamada "distancia". Utiliza la fila y la columna como las coordenadas de cada punto.
3.   Carga el archivo fabrica2-c.csv como un Dataframe. Agrupa por columnas x y por tipo, agrega el coste y calcula la suma del coste.

Utiliza `print` para mostrar los resultados por la pantalla.

⛔ Prohibido: Este ejercicio debe utilizarse exclusivamente la librería PANDAS, NO será correcta la programación en Python directa para obtener estos datos.

**¿QUÉ TENGO QUE HACER?**

Al final de este notebook, hay  3 celdas de código donde debes resolver estos tres problemas con tus conocimientos de Pandas.


In [1]:
# ENTORNO CLIPS
#!pip install clipspy

import pandas as pd

import clips

env = clips.Environment()

# TEMPLATES y FACTS
## CASILLA
La fábrica se compone de un conjunto de casillas unas junto a otras. Cada Casilla puede ser ocupada por el Agente o bien por algún otro elemento del juego.



*   Posición x: la fila , comenzaremos por 1.
*   Posición y: la columna, comenzaremos por 1.
*   tipo: el elemento que lo ocupa:
      *   "-" (el guión) representará que está vacía.
      *   "X" (X mayúscula) representará un elemento, como una máquina, que ocupa una casilla y que no puede ser ocupada por el agente.
      *   "X" (X mayúscula) representará que el agente ya la ha visitado y no puede volver a usarse.
      *  "B" se encuentra la nueva batería.
*   valor: este campo auxiliar numérico permite añadir información extra cuando haga falta.
*   coste: un valor numérico que indica un valor de coste de alcanzar cada casilla.



## AGENTE
Nuestro robot que intenta alcanzar la batería de repuesto.
*   Posición x: la fila.
*   Posición y: la columna.



## SENSOR
Nuestro agente dispone de una serie de sensores que analizan posibles casillas donde podría llegar a moverse. Los *Sensores* representan esas potenciales casillas.
*   Posición x: la fila.
*   Posición y: la columna.

## FRONTERA
Nuestro agente puede moverse a aquellas casillas que el sensor ha detectado y que además nos hemos asegurado que están libres para movernos.

La *Frontera* representa el conjunto de casillas ya detectadas por los sensores y que además están disponibles para que el agente avance.
Por ejemplo: un sensor puede detectar una casilla dentro de la fábrica, pero que ya haya sido visitada ('X'). En ese caso, no se tratará de una casilla *Frontera*.

## DESACTIVA FRONTERA
Una vez que el agente ha dedicido dónde moverse entre los candidados disponibles de la frontera, el agente se desplazará a esa casilla.

Una vez que el agente se ha movido, es necesario borrar los elementos de la frontera, para poder ha explorar nuevas casillas.

*DesactivaFrontera* es un elemento auxiliar que permite borrar la frontera.

In [2]:
# TEMPLATES

# Template tablero:
#   este template representa...
#     ... aquí los comentarios
templateCasilla = """(deftemplate Casilla
  (slot x (type INTEGER))
  (slot y (type INTEGER))
  (slot tipo (type STRING))
  (slot valor (type INTEGER))
  (slot coste (type INTEGER))
  )"""

templateAgente = """(deftemplate Agente
  (slot x (type INTEGER))
  (slot y (type INTEGER))
  )"""

templateSensor = """(deftemplate Sensor
  (slot x (type INTEGER))
  (slot y (type INTEGER))
  )"""

templateFrontera = """(deftemplate Frontera
  (slot x (type INTEGER))
  (slot y (type INTEGER))
  )"""

templateDesactivaFrontera = """(deftemplate DesactivaFrontera
  )"""

templateOBJETIVO = """(deftemplate OBJETIVOALCANZADO
  )"""



# AÑADIR LOS TEMPLATES AL ENTORNO
env.build(templateCasilla)
env.build(templateAgente)
env.build(templateSensor)
env.build(templateFrontera)
env.build(templateDesactivaFrontera)
env.build(templateOBJETIVO)



templateCasilla = env.find_template('Casilla')
templateAgente = env.find_template('Agente')
templateSensor = env.find_template('Sensor')
templateFrontera = env.find_template('Frontera')
templateKillFrontera = env.find_template('DesactivaFrontera')
templateOBJETIVO = env.find_template('OBJETIVOALCANZADO')

# ENTRADA: la fábrica
El sistema carga desde un fichero .csv la información de la fábrica y los peligros que se encuentran.

El fichero .csv almacena en cada fila la información de la casilla.

Inicialmente los ficheros .csv contendrán la información de una fábrica de 12 casillas: 3 filas y 4 columnas.

La cabecera corresponde a:


*   x: la posición x de la casilla
*   y: la posición y de la casilla
*   t: (tipo) qué hay en la casilla: (X) corresponde a una máquina, (B) la batería
*  v: (valor) el valor
*  c: (coste) en ocasiones almacena un dato sobre el coste de alcanzar la casilla (un valor inventado sobre el gasto que realiza el agente en la acción de desplazarse a una casilla)





## Función cargarCSV()
La función cargarCSV() sirve para cargar la fábrica desde un fichero .csv

cargarCSV() recibe como argumentos:


*   file: la cadena de texto con el nombre del fichero .csv a cargar
*   templateCasilla: un template ya creado de templateCasilla

La función cargarCSV() hace lo siguiente:

1.   Inserta en el environment de CLIPS los facts de tipo templateCasilla con la información disponible en el fichero .csv
2.   Devolverá el dataframe con los datos cargados desde el fichero .csv






In [3]:
# La función cargarCSV dato un nombre de fichero y un template templateCasilla
#  1. Carga el fichero en un dataframe
#  2. Para cada fila del dataframe, extrae los datos y crea un fact de templateCasilla en el environment
#  3. Devuelve el dataframe con los datos cargados.
def cargarCSV(file, templateCasilla):
  df = pd.read_csv(file)
  for index, row in df.iterrows():
    x1=row['x']
    y1=row['y']
    t1=row['t']
    v1=row['v']
    c1=row['c']
    templateCasilla.assert_fact(x=x1,y=y1,tipo=t1,valor=v1,coste=c1)
  return df

## EJEMPLO DE CARGA, imprime los facts creados y muestra los datos del dataframe cargado
df_fabrica = cargarCSV('../Documentos/fabrica1.csv', templateCasilla)

print('Listado de facts:')
for f in env.facts():
  print(f)

df_fabrica.head(3) #muestra los 3 primeras filas del dataframe

df_fabrica

FileNotFoundError: [Errno 2] No such file or directory: '../Documentos/fabrica1.csv'

# FACTS DE INICIO

Una vez cargada la fábrica, hay que establecer dónde se encuentra nuestro agente.

Por defecto el agente se encuentra en la casilla x=1 y=1.


In [ ]:
#FACTS INICIO

factStart = templateAgente.assert_fact(x=1, y=1)


# MOSTRAMOS LOS FACTS DISPONIBLES EN LA BASE DE DATOS
print('Listado de facts:')
for f in env.facts():
  print(f)

Listado de facts:
(Casilla (x 1) (y 1) (tipo "-") (valor 0) (coste 0))
(Casilla (x 1) (y 2) (tipo "-") (valor 0) (coste 0))
(Casilla (x 1) (y 3) (tipo "-") (valor 0) (coste 0))
(Casilla (x 2) (y 1) (tipo "-") (valor 0) (coste 0))
(Casilla (x 2) (y 2) (tipo "-") (valor 0) (coste 0))
(Casilla (x 2) (y 3) (tipo "-") (valor 0) (coste 0))
(Casilla (x 3) (y 1) (tipo "-") (valor 0) (coste 0))
(Casilla (x 3) (y 2) (tipo "X") (valor 0) (coste 0))
(Casilla (x 3) (y 3) (tipo "-") (valor 0) (coste 0))
(Casilla (x 4) (y 1) (tipo "-") (valor 0) (coste 0))
(Casilla (x 4) (y 2) (tipo "-") (valor 0) (coste 0))
(Casilla (x 4) (y 3) (tipo "B") (valor 0) (coste 0))
(Agente (x 1) (y 1))


# REGLAS BÁSICAS DE FUNCIONAMIENTO ➡

Disponemos de un conjunto elemental de reglas que permiten de forma básica que el Agente explore la fábrica para poder alcanzar la batería.


## Regla RuleObjetivo
Esta regla comprueba que el agente se encuenta en la casilla donde se ubica la batería.
Si es así, ¡habremos alcanzado nuestro objetivo!

SI (Agente en el mismo sitio que la batería) => OBJETIVO ALCANZADO


## Regla RuleActivaSensor
Si el Agente no se encuentra en el lugar de la batería, activaremos los Sensores para explorar la fábrica.

Tenemos 4 tipos de sensores para las posiciones adyacentes al Agente: arriba, abajo, izquierda y derecha.

SI (no OBJETIVO) => crear los 4 sensores adyacentes.


## Regla RuleDescartaSensorFuera

Si detectamos que un sensor "se sale" de la fábrica, entonces lo descartamos.

Un sensor se sale de la fábrica si analiza una posición donde no hay una casilla.

SI (sensor no tiene casilla) => eliminar el sensor

## Regla RuleDescartaSensorOcupado

Si detectamos que un sensor de una casilla de la fábrica que no es utilizable (por ejemplo, que ya esté visitado).

SI (casilla del sensor no utilizable) => eliminar el sensor

## Regla RuleActivarFrontera

Consideramos frontera todas aquellas casillas que los sensores han detectado que son visitables por el agente.

SI (casilla del sensor utilizable) =>
      borrar el sensor y crear la frontera para esa casilla.


## Regla AvanzarBasico

Una vez que el Agente tiene activadas las fronteras, se moverá a una de ellas.

Antes de moverse, el agente debe esperar que se hayan analizado todos los sensores y, por tanto, ya se dispone de todas las fronteras.

Al ser un movimiento básico (poco inteligente) elegirá una frontera sin ningún criterio en especial.

Al moverse, la casilla donde se encuentra pasará a estar visitada.

Una vez que se haya movido el agente, habrá que desactivar la frontera para permitir una nueva búsqueda más adelante.


SI (no hay sensores Y frontera disponible) =>
hacer estos 4 pasos:          
*   borrar la frontera.
*   casilla actual del agente pasa a estar visitada "X".
*   mover al agente a la casilla indicada por la frontera.
*   ordenar la desactivación de la frontera.

## Reglas de desactivación de la frontera
La desactivación de la frontera es un proceso de dos pasos para la eliminación de todas las Frontera que haya en ese momento. Se define mediante dos reglas.

1.   *DesactivaFronteraStart*: busca cualquier *Frontera* y la elimina.
2.  *DesactivaFronteraStop*: cuando ya no hay más fronteras se para el proceso de desactivación de la frontera.


In [ ]:
#RULES

ruleObjetivo = """
(defrule RuleObjetivo
  (declare (salience 1000))
  (Casilla (x ?x)(y ?y)(tipo "B"))
  (Agente (x ?x)(y ?y))
  =>
  (assert (OBJETIVOALCANZADO ))
)
"""

# A partir de la posición del agente, se crean los sensores en las posiciones adyacentes (arriba, abajo, iz, der)
ruleActivaSensor = """
(defrule RuleActivaSensor
  (not (exists (OBJETIVOALCANZADO )))
  (Agente (x ?x) (y ?y))
  =>
  (assert (Sensor (x ?x ) (y (+ ?y 1)) ))
  (assert (Sensor (x ?x ) (y (- ?y 1)) ))
  (assert (Sensor (x (+ ?x 1)) (y ?y) ))
  (assert (Sensor (x (- ?x 1)) (y ?y) ))
)
"""
# Se descartan los sensores que queden fuera de la fábrica (que no haya casilla)
ruleDescartaSensorFuera = """
(defrule RuleDescartaSensorFuera
  ?s <-(Sensor (x ?x) (y ?y))
  (not (exists (Casilla (x ?x) (y ?y))))
  =>
  (retract ?s)
)
"""
# Se descartan los sensores que detectan casillas ya visitadas (que haya X en la casilla)
ruleDescartaSensorOcupado = """
(defrule RuleDescartaSensorOcupado
  ?s <-(Sensor (x ?x) (y ?y))
  (Casilla (x ?x) (y ?y) (tipo "X"))
  =>
  (retract ?s)
)
"""
# Regla para seleccionar sensor e incluirlos en la frontera de casillas no "X"
ruleActivarFrontera = """
(defrule RuleActivarFrontera
  ?s <-(Sensor (x ?x) (y ?y))
  ?c <-(Casilla (x ?x) (y ?y) (tipo ?t))
  (not (eq ?t "X"))
  =>
  (retract ?s)
  (assert (Frontera (x ?x) (y ?y) ))
)
"""


# Regla Avance Básico: si hay una frontera cuya casilla no "X" -> avanza actual a esa casilla.
ruleAvanzarBasico = """
(defrule RuleAvanzarBasico
  (not (exists (Sensor )))
  ?f <-(Frontera (x ?x) (y ?y))
  ?c2 <-(Casilla (x ?x) (y ?y) (tipo ?t))
  (not (eq ?t "X"))
  ?a <-(Agente (x ?i) (y ?j))
  ?c1 <-(Casilla (x ?i) (y ?j))
  =>

  (retract ?f)

  (modify ?c1 (tipo "X") )

  (retract ?a)
  (assert (Agente (x ?x) (y ?y) ))

  (assert (DesactivaFrontera))

  
)
"""

ruleDesactivaFronteraStart = """
(defrule RuleDesactivaFronteraStart
  (declare (salience 90))
  (DesactivaFrontera)
  ?f <-(Frontera (x ?x) (y ?y))
  =>
  (retract ?f)
)
"""
ruleDesactivaFronteraStop = """
(defrule RuleDesactivaFronteraStop
  ?front <-(DesactivaFrontera)
  (not (exists (Frontera )))
  =>
  (retract ?front)
)
"""

# añado  REGLA en la BASE DE REGLAS
env.build(ruleObjetivo)
env.build(ruleActivaSensor)
env.build(ruleDescartaSensorFuera)
env.build(ruleDescartaSensorOcupado)
env.build(ruleActivarFrontera)
env.build(ruleAvanzarBasico)
env.build(ruleDesactivaFronteraStart)
env.build(ruleDesactivaFronteraStop)

# PROGRAMA PRINCIPAL ¡¡ QUE COMIENCE EL JUEGO !! 🎉

Una vez listo la fábrica, el agente está colocado en la casilla de inicio y tenemos las reglas de búsqueda...

**PROGRAMA PRINCIPAL: el agente A empieza la búsqueda** 👁

In [ ]:
# LANZO EL MOTOR DE RAZONAMIENTO

env.run()

# MOSTRAMOS RESULTADOS TRAS EJECUTAR REGLAS
print('Facts tras ejecutar las reglas')
for f in env.facts():
    print('Indice fact', f.index,':',f)

Facts tras ejecutar las reglas
Indice fact 1 : (Casilla (x 1) (y 1) (tipo "X") (valor 0) (coste 0))
Indice fact 2 : (Casilla (x 1) (y 2) (tipo "X") (valor 0) (coste 0))
Indice fact 3 : (Casilla (x 1) (y 3) (tipo "X") (valor 0) (coste 0))
Indice fact 4 : (Casilla (x 2) (y 1) (tipo "X") (valor 0) (coste 0))
Indice fact 5 : (Casilla (x 2) (y 2) (tipo "X") (valor 0) (coste 0))
Indice fact 6 : (Casilla (x 2) (y 3) (tipo "X") (valor 0) (coste 0))
Indice fact 7 : (Casilla (x 3) (y 1) (tipo "X") (valor 0) (coste 0))
Indice fact 8 : (Casilla (x 3) (y 2) (tipo "X") (valor 0) (coste 0))
Indice fact 9 : (Casilla (x 3) (y 3) (tipo "-") (valor 0) (coste 0))
Indice fact 10 : (Casilla (x 4) (y 1) (tipo "X") (valor 0) (coste 0))
Indice fact 11 : (Casilla (x 4) (y 2) (tipo "X") (valor 0) (coste 0))
Indice fact 12 : (Casilla (x 4) (y 3) (tipo "B") (valor 0) (coste 0))
Indice fact 78 : (Agente (x 4) (y 3))
Indice fact 80 : (OBJETIVOALCANZADO)


## CELDAS RESPUESTA DEL EJERCICIO 1

### Apartado 1

**Guarda y muestra el valor de las coordenadas donde se encuentra la batería.**

In [ ]:
coordenadas_X = df_fabrica[df_fabrica['t'] == 'B']["x"].values[0] 
coordenadas_Y = df_fabrica[df_fabrica['t'] == 'B']["y"].values[0]
print(f"Coordenadas => x:{coordenadas_X}, y: {coordenadas_Y}")


Coordenadas => x:4, y: 3


### Apartado 2

**Calcula la distancia entre cada casilla de la fábrica y la posición de la batería y guarda los valores en una nueva columna del dataframe llamada "distancia". Utiliza la fila y la columna como las coordenadas de cada punto.**



In [ ]:
import math
distancias = []
for i in range(1, 5):
    for j in range(1, 4):
        distancia = math.sqrt((i-coordenadas_X)**2+(j-coordenadas_Y)**2)
        print(f"La distancia de la coordenada ({i},{j}) a la bateria es {distancia}")
        distancias.append(distancia)

df_fabrica["distancia"] = distancias[:len(df_fabrica)]
df_fabrica


La distancia de la coordenada (1,1) a la bateria es 3.605551275463989
La distancia de la coordenada (1,2) a la bateria es 3.1622776601683795
La distancia de la coordenada (1,3) a la bateria es 3.0
La distancia de la coordenada (2,1) a la bateria es 2.8284271247461903
La distancia de la coordenada (2,2) a la bateria es 2.23606797749979
La distancia de la coordenada (2,3) a la bateria es 2.0
La distancia de la coordenada (3,1) a la bateria es 2.23606797749979
La distancia de la coordenada (3,2) a la bateria es 1.4142135623730951
La distancia de la coordenada (3,3) a la bateria es 1.0
La distancia de la coordenada (4,1) a la bateria es 2.0
La distancia de la coordenada (4,2) a la bateria es 1.0
La distancia de la coordenada (4,3) a la bateria es 0.0


,x,y,t,v,c,distancia
0,1,1,-,0,0,3.605551
1,1,2,-,0,0,3.162278
2,1,3,-,0,0,3.000000
3,2,1,-,0,0,2.828427
4,2,2,-,0,0,2.236068
5,2,3,-,0,0,2.000000
6,3,1,-,0,0,2.236068
7,3,2,X,0,0,1.414214
8,3,3,-,0,0,1.000000
9,4,1,-,0,0,2.000000


### Apartado 3

**Carga el archivo fabrica2-c.csv como un Dataframe. Agrupa por columnas x y por tipo, agrega el coste y calcula la suma del coste.**

In [ ]:
datos_fab = pd.read_csv("../Documentos/fabrica2-c.csv")
r = datos_fab.groupby(["x",'t'])['c'].aggregate(['sum'])
r

sum
x t     
1 -    9
2 -    6
3 -    7
  X    4
4 -    3
  B    4